<div align="center">

# NB4 · Attack 1: Suppressing CGM Hypoglycaemia Alerts

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · 28 October 2026 · Session 11:00 to 11:30

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB4_Attack_CGM_Hypoglycemia.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

A continuous glucose monitor (CGM) reports interstitial glucose every five minutes, and modern systems raise a predictive alert before glucose falls below 70 mg/dL. For older adults with long-standing type 1 diabetes, who often lose the warning symptoms of hypoglycaemia, that alert can be the only warning they get. This notebook builds such an alert model on real CGM recordings, attacks it with a query-efficient decision-based attack, and asks what explanations reveal about the manipulation.

**Data.** The recordings come from 200 adults aged 60 and older with type 1 diabetes who wore a blinded CGM for about two weeks in a study of severe hypoglycaemia risk factors [4]. The notebook downloads the curated copy distributed by the GlucoBench project [5], pinned to a fixed commit of its GitHub repository. GlucoBench lists this dataset under CC BY-SA 4.0. The analyses here are those of the workshop author and have not been reviewed by the original study group. Timestamps in the file are de-identified and start on an artificial date.

**Threat model.** The attacker can alter the glucose values that reach the alert logic, for example through a compromised phone application, a relay between sensor and phone, or a malicious third-party integration. Each reading may be shifted by at most 15 mg/dL, a change within the error band that users already accept from CGM sensors. The attacker cannot see the model, its parameters or its risk score, and observes only whether the alert fires. Medical device cybersecurity advisories have covered exactly this kind of wireless path in diabetes devices [11].

The full run takes three to five minutes on a free Colab CPU.

In [ ]:
import importlib.util, subprocess, sys
PACKAGES = {"shap": "shap>=0.46", "gemex": "gemex>=1.2.2"}
missing = [spec for module, spec in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import shap
import ipywidgets as widgets
from gemex import Explainer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

## 1. Real CGM data

In [ ]:
GLUCOBENCH = "https://raw.githubusercontent.com/IrinaStatsLab/GlucoBench/"
COMMIT = "661d840a98b316df51faa13a7100430afcbbb5b7"
zip_path = fetch([GLUCOBENCH + COMMIT + "/raw_data.zip", GLUCOBENCH + "main/raw_data.zip"],
                 DATA_DIR / "glucobench_raw_data.zip", timeout=300)
if hashlib.md5(zip_path.read_bytes()).hexdigest() != "49667c2e7e14640d0e4089041ca9e948":
    print("Note: the archive differs from the pinned GlucoBench release.")
with zipfile.ZipFile(zip_path) as z, z.open("raw_data/weinstock.csv") as fh:
    cgm = pd.read_csv(fh, usecols=["id", "gl", "time"])
cgm["time"] = pd.to_datetime(cgm["time"], format="%Y-%m-%d %H:%M:%S")
cgm = cgm.sort_values(["id", "time"]).reset_index(drop=True)
if QUICK:
    cgm = cgm[cgm["id"].isin(np.sort(cgm["id"].unique())[:60])]

per_patient = cgm.groupby("id")["gl"].agg(below70=lambda g: (g < 70).mean(), below54=lambda g: (g < 54).mean())
days = cgm.groupby("id")["time"].agg(lambda t: (t.max() - t.min()).total_seconds() / 86400)
print(f"{cgm['id'].nunique()} patients · {len(cgm):,} readings · median {days.median():.1f} days per patient")
print(f"readings below 70 mg/dL: {(cgm['gl'] < 70).mean():.1%} · below 54 mg/dL: {(cgm['gl'] < 54).mean():.1%}")
print(f"patients above the 1% time-below-70 target for older, high-risk adults [6]: {(per_patient['below70'] > 0.01).mean():.0%}")

In [ ]:
pid0 = per_patient["below70"].idxmax()
trace = cgm[cgm["id"] == pid0].copy()
hours = (trace["time"] - trace["time"].iloc[0]).dt.total_seconds() / 3600
sel = hours < 72
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.axhspan(70, 180, color=SAFE, alpha=0.08, label="target range 70 to 180 mg/dL")
ax.plot(hours[sel], trace["gl"][sel], color=INK, lw=1)
low = sel & (trace["gl"] < 70)
ax.scatter(hours[low], trace["gl"][low], color=ALERT, s=8, zorder=3, label="below 70 mg/dL")
ax.set_xlabel("hours since the start of the recording")
ax.set_ylabel("glucose (mg/dL)")
ax.set_title(f"Three days of CGM from one participant ({per_patient.loc[pid0, 'below70']:.0%} of readings below 70)")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

## 2. A predictive low-glucose alert

Each training example is one hour of CGM (12 readings) cut from a stretch without gaps. The alert should fire if any of the next six readings, that is the next 30 minutes, falls below 70 mg/dL. Windows whose latest reading is already below 70 are excluded, because at that point a simple threshold alarm takes over. Ten summary features describe the hour: the current value, level and spread, and the short-term trend. A histogram gradient boosting model learns the alert, and the patients are split 70 to 30 so that no patient appears in both training and test data.

In [ ]:
PAST, HORIZON, STRIDE = 12, 6, 3
FEATURE_NAMES = ["glucose_now", "mean_60min", "min_60min", "max_60min", "sd_60min",
                 "slope_15min", "slope_30min", "change_5min", "change_60min", "trend_change"]


def make_windows(frame):
    past, future, pids = [], [], []
    for pid, g in frame.groupby("id", sort=False):
        minutes = g["time"].to_numpy().astype("datetime64[s]").astype(np.int64) / 60.0
        values = g["gl"].to_numpy(float)
        gaps = np.diff(minutes)
        breaks = np.where((gaps < 2.5) | (gaps > 7.5))[0] + 1
        for seg in np.split(values, breaks):
            if len(seg) < PAST + HORIZON:
                continue
            W = np.lib.stride_tricks.sliding_window_view(seg, PAST + HORIZON)[::STRIDE]
            past.append(W[:, :PAST])
            future.append(W[:, PAST:])
            pids.append(np.full(len(W), pid))
    past, future, pids = np.vstack(past), np.vstack(future), np.concatenate(pids)
    keep = past[:, -1] >= 70
    return past[keep], future[keep], (future[keep].min(axis=1) < 70).astype(int), pids[keep]


def _slope(block):
    t = np.arange(block.shape[1]) * 5.0
    t -= t.mean()
    return (block - block.mean(axis=1, keepdims=True)) @ t / (t @ t)


def cgm_features(W):
    """Ten features of a one-hour CGM window (readings in mg/dL, oldest first); slopes in mg/dL per minute."""
    W = np.atleast_2d(np.asarray(W, dtype=float))
    s15, s30, s15_before = _slope(W[:, -4:]), _slope(W[:, -7:]), _slope(W[:, -7:-3])
    return np.column_stack([W[:, -1], W.mean(1), W.min(1), W.max(1), W.std(1),
                            s15, s30, (W[:, -1] - W[:, -2]) / 5.0, W[:, -1] - W[:, 0], s15 - s15_before])


Xw, fut, yw, pid = make_windows(cgm)
patients = np.unique(pid)
test_patients = rng.choice(patients, size=int(0.3 * len(patients)), replace=False)
te = np.isin(pid, test_patients)
Xw_tr, y_tr, pid_tr = Xw[~te], yw[~te], pid[~te]
Xw_te, y_te, fut_te = Xw[te], yw[te], fut[te]
print(f"{len(Xw):,} windows · {yw.mean():.1%} followed by glucose below 70 within 30 minutes")
print(f"train: {len(Xw_tr):,} windows from {len(patients) - len(test_patients)} patients · "
      f"test: {len(Xw_te):,} windows from {len(test_patients)} patients")

The alert threshold is set on out-of-fold predictions for the training patients so that 85 percent of the coming lows are announced, a typical design target that trades sensitivity against alarm fatigue.

In [ ]:
def new_model():
    return make_pipeline(FunctionTransformer(cgm_features),
                         HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=31,
                                                        l2_regularization=1.0, random_state=SEED))


oof = np.zeros(len(Xw_tr))
for fit_idx, val_idx in GroupKFold(n_splits=3).split(Xw_tr, y_tr, pid_tr):
    oof[val_idx] = new_model().fit(Xw_tr[fit_idx], y_tr[fit_idx]).predict_proba(Xw_tr[val_idx])[:, 1]
tau = float(np.quantile(oof[y_tr == 1], 0.15))
alert_model = new_model().fit(Xw_tr, y_tr)
p_te = alert_model.predict_proba(Xw_te)[:, 1]
alert_te = p_te >= tau
tp, fp = (alert_te & (y_te == 1)).sum(), (alert_te & (y_te == 0)).sum()
pd.Series({"alert threshold (risk)": tau, "test AUROC": roc_auc_score(y_te, p_te),
           "sensitivity": tp / (y_te == 1).sum(), "specificity": 1 - fp / (y_te == 0).sum(),
           "positive predictive value": tp / max(tp + fp, 1)}).round(3).to_frame("value")

Before attacking the model it is worth knowing what it relies on. TreeSHAP values, in log-odds units, show that the current glucose level and the short-term slope dominate, which matches clinical intuition.

In [ ]:
hgb = alert_model[-1]
tree_explainer = shap.TreeExplainer(hgb)
sample = rng.choice(len(Xw_te), size=min(2000, len(Xw_te)), replace=False)
sv = tree_explainer(pd.DataFrame(cgm_features(Xw_te[sample]), columns=FEATURE_NAMES))
shap.plots.beeswarm(sv, max_display=10, show=False)
plt.title("What drives the hypoglycaemia alert (TreeSHAP, log-odds)")
plt.show()

### The attacker's toolkit

The function below implements a decision-based attack in the spirit of HopSkipJump [1]. The attacker never sees the model, its weights or its scores. The only thing observed is the final decision: alert or no alert. The attack proceeds in four moves that repeat:

1. Find any manipulated input without an alert, inside the permitted perturbation box.
2. Walk back towards the genuine input with a binary search until the decision boundary is reached.
3. Estimate the direction of the boundary by probing random nearby points and recording only the yes or no answers.
4. Step along that direction, then search back towards the genuine input again.

Every query is counted, because in a real system each query is an interaction that a defender could, in principle, observe.

In [ ]:
def hsja_attack(decide, x0, lower, upper, init_candidates, n_iter=15, n_dirs=60, seed=0):
    """Decision-based attack in the spirit of HopSkipJump (l2, untargeted, box-constrained).

    decide(X) returns True where the model raises the alert. The search stays inside [lower, upper].
    Returns a dict with the manipulated input (None if no alert-free start exists), distances and query count.
    """
    rs = np.random.default_rng(seed)
    x0, lower, upper = (np.asarray(a, dtype=float) for a in (x0, lower, upper))
    queries = 0

    def alert(X):
        nonlocal queries
        X = np.atleast_2d(X)
        queries += len(X)
        return np.asarray(decide(X), dtype=bool)

    cands = np.clip(np.atleast_2d(np.asarray(init_candidates, dtype=float)), lower, upper)
    free = ~alert(cands)
    if not free.any():
        return dict(x_adv=None, success=False, queries=queries, l2=np.nan, linf=np.nan)
    x_adv = cands[free][np.argmin(np.linalg.norm(cands[free] - x0, axis=1))]

    def to_boundary(x_free, tol=1e-3):
        lo, hi = 0.0, 1.0            # hi always gives an alert-free point on the segment x0 -> x_free
        while hi - lo > tol:
            mid = 0.5 * (lo + hi)
            if alert(x0 + mid * (x_free - x0))[0]:
                lo = mid
            else:
                hi = mid
        return x0 + hi * (x_free - x0)

    xb = to_boundary(x_adv)
    for t in range(1, n_iter + 1):
        dist = np.linalg.norm(xb - x0)
        if dist < 1e-6:
            break
        radius = max(0.1 * dist / np.sqrt(t), 1e-3)
        U = rs.normal(size=(n_dirs, xb.size))
        U /= np.linalg.norm(U, axis=1, keepdims=True)
        phi = np.where(alert(np.clip(xb + radius * U, lower, upper)), -1.0, 1.0)
        v = ((phi - phi.mean())[:, None] * U).mean(axis=0) if phi.std() > 0 else phi[0] * U.mean(axis=0)
        if np.linalg.norm(v) < 1e-12:
            break
        v /= np.linalg.norm(v)
        step = dist / np.sqrt(t)
        for _ in range(12):
            cand = np.clip(xb + step * v, lower, upper)
            if not alert(cand)[0]:
                xb = to_boundary(cand)
                break
            step /= 2
    return dict(x_adv=xb, success=True, queries=queries,
                l2=float(np.linalg.norm(xb - x0)), linf=float(np.abs(xb - x0).max()))

### The defender's toolkit

Three tools turn explanations into security signals. The first is an anomaly score on explanation vectors: a Mahalanobis distance measuring how unusual the SHAP attribution pattern of an input is, compared with genuine inputs that received the same decision. The second is a counterfactual margin: the smallest change in a clinically meaningful direction that would make the alert fire again. It is a counterfactual explanation in the sense of NB2, applied to a "no alert" decision, and it answers the question of how close that decision was to flipping. The third is the Page-Hinkley test [2], a sequential change detector that raises an alarm when the running mean of a monitored signal drifts upwards. Applied to a stream of scores, it looks for a sustained campaign rather than a single suspicious input.

In [ ]:
class MahalanobisDetector:
    """Distance of a vector from a reference population (mean and regularised covariance)."""

    def fit(self, Z):
        Z = np.asarray(Z, dtype=float)
        self.mu = Z.mean(axis=0)
        cov = np.cov(Z, rowvar=False) + 1e-6 * np.eye(Z.shape[1])
        self.precision = np.linalg.pinv(cov)
        return self

    def score(self, Z):
        D = np.asarray(Z, dtype=float) - self.mu
        return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", D, self.precision, D), 0.0))


def page_hinkley(stream, delta, lam):
    """Page-Hinkley test for an upward shift in the mean. Returns (statistic trace, index of first alarm or None)."""
    mean, cum, cum_min, trace, alarm = 0.0, 0.0, 0.0, [], None
    for t, x in enumerate(stream, start=1):
        mean += (x - mean) / t
        cum += x - mean - delta
        cum_min = min(cum_min, cum)
        trace.append(cum - cum_min)
        if alarm is None and cum - cum_min > lam:
            alarm = t - 1
    return np.array(trace), alarm


def counterfactual_margin(decide, X, move, max_step=1.0, tol=0.01):
    """Smallest step s in [0, max_step] for which decide(move(X, s)) raises the alert; max_step if none does.

    move(X, s) applies a step of size s[i] to row i in a clinically meaningful direction.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    lo, hi = np.zeros(len(X)), np.full(len(X), float(max_step))
    active = np.asarray(decide(move(X, hi)), dtype=bool)
    while np.any(active & (hi - lo > tol)):
        mid = 0.5 * (lo + hi)
        fire = np.asarray(decide(move(X, mid)), dtype=bool)
        hi = np.where(active & fire, mid, hi)
        lo = np.where(active & ~fire, mid, lo)
    return np.where(active, hi, float(max_step))


def detection_auc(scores_clean, scores_attacked):
    """AUROC of a score that should be higher for manipulated inputs."""
    from sklearn.metrics import roc_auc_score
    s = np.concatenate([scores_clean, scores_attacked])
    lab = np.concatenate([np.zeros(len(scores_clean)), np.ones(len(scores_attacked))])
    return roc_auc_score(lab, s)


def campaign(clean_scores, attacked_scores, share=0.2, L=400, delta=0.25, seed=42):
    """Simulate a monitoring stream that switches halfway to a partly manipulated state and run Page-Hinkley.

    Scores are standardised with the median and interquartile range of genuine scores; the alarm threshold is
    set just above the largest statistic seen on 30 streams without any attack.
    """
    rs = np.random.default_rng(seed)
    clean_scores, attacked_scores = np.asarray(clean_scores, float), np.asarray(attacked_scores, float)
    med = np.median(clean_scores)
    iqr = np.subtract(*np.quantile(clean_scores, [0.75, 0.25])) + 1e-9
    z = lambda v: (np.asarray(v) - med) / iqr
    lam = 1.1 * max(page_hinkley(z(rs.choice(clean_scores, L)), delta, np.inf)[0].max() for _ in range(30))
    stream = rs.choice(clean_scores, L)
    t0 = L // 2
    hit = rs.random(L - t0) < share
    stream[t0:][hit] = rs.choice(attacked_scores, hit.sum())
    trace, alarm = page_hinkley(z(stream), delta, lam)
    return z(stream), trace, lam, alarm, t0


def plot_campaigns(signals, share=0.2):
    """signals: list of (title, genuine scores, manipulated scores); higher scores must mean more suspicious."""
    fig, axes = plt.subplots(1, len(signals), figsize=(6.5 * len(signals), 3.4), squeeze=False)
    for ax, (title, sc, sa) in zip(axes[0], signals):
        z_stream, trace_ph, lam, alarm, t0 = campaign(sc, sa, share=share)
        ax.scatter(np.arange(len(z_stream)), z_stream, s=5, alpha=0.5,
                   color=np.where(np.arange(len(z_stream)) >= t0, ALERT, INK))
        ax.axvline(t0, color=ALERT, ls="--", lw=1, label="campaign starts")
        if alarm is not None:
            ax.axvline(alarm, color=AMBER, lw=2, label=f"Page-Hinkley alarm after {alarm - t0} inputs")
        else:
            ax.text(0.02, 0.92, "no alarm", transform=ax.transAxes, color=AMBER, fontweight="bold")
        ax.set_title(title)
        ax.set_xlabel("near-threshold input in the stream")
        ax.set_ylabel("standardised score")
        ax.legend(frameon=False, fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()

## 3. The attack

The attack targets windows in which the alert fired correctly: a low really followed within 30 minutes. For each one, the attacker searches for the smallest change, at most 15 mg/dL per reading and within the sensor range of 40 to 400 mg/dL, that silences the alert. Starting points include raising every reading by the full budget and a ramp that raises recent readings more than old ones.

In [ ]:
BUDGET = 15.0


def decide(W):
    return alert_model.predict_proba(W)[:, 1] >= tau


tp_idx = np.where((y_te == 1) & alert_te)[0]
attack_idx = rng.choice(tp_idx, size=min(len(tp_idx), 12 if QUICK else 60), replace=False)
results = []
t_start = time.time()
for k, i in enumerate(attack_idx):
    x0 = Xw_te[i]
    lo, hi = np.maximum(x0 - BUDGET, 40), np.minimum(x0 + BUDGET, 400)
    init = np.vstack([hi, x0 + BUDGET * np.linspace(0, 1, PAST), rng.uniform(lo, hi, size=(100, PAST))])
    r = hsja_attack(decide, x0, lo, hi, init, n_iter=8 if QUICK else 15, n_dirs=40 if QUICK else 60, seed=k)
    r.update(index=i, x0=x0, future_min=fut_te[i].min())
    results.append(r)
att = pd.DataFrame(results)
ok = att["success"].to_numpy(bool)
print(f"{len(att)} correct alerts attacked in {time.time() - t_start:.0f} s")
pd.Series({"alerts suppressed": f"{ok.mean():.0%}",
           "median largest change per reading (mg/dL)": round(att.loc[ok, "linf"].median(), 1),
           "median total change, l2 (mg/dL)": round(att.loc[ok, "l2"].median(), 1),
           "median queries": int(att.loc[ok, "queries"].median()) if ok.any() else 0,
           "lowest glucose that followed a suppressed alert": f"{att.loc[ok, 'future_min'].min():.0f} mg/dL" if ok.any() else "none"}
          ).to_frame("result")

In [ ]:
show = att[ok].sort_values("future_min").head(2)
fig, axes = plt.subplots(1, len(show), figsize=(12, 3.6), squeeze=False)
t_past, t_future = np.arange(-55, 1, 5), np.arange(5, 31, 5)
for ax, (_, r) in zip(axes[0], show.iterrows()):
    ax.fill_between(t_past, r["x0"] - BUDGET, r["x0"] + BUDGET, color=MUTED, alpha=0.15, label="attack budget")
    ax.plot(t_past, r["x0"], "o-", color=INK, ms=3, label=f"true CGM (risk {alert_model.predict_proba(r['x0'][None])[0, 1]:.2f})")
    ax.plot(t_past, r["x_adv"], "o-", color=ALERT, ms=3,
            label=f"reported CGM (risk {alert_model.predict_proba(r['x_adv'][None])[0, 1]:.2f})")
    ax.plot(t_future, fut_te[r["index"]], "s--", color=AMBER, ms=3, label="what happened next")
    ax.axhline(70, color=ALERT, lw=0.8, ls=":")
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("minutes relative to the alert decision")
    ax.set_ylabel("glucose (mg/dL)")
    ax.legend(frameon=False, fontsize=7)
fig.suptitle(f"Suppressed alerts (threshold risk {tau:.2f})", y=1.02)
plt.tight_layout()
plt.show()

Two further observations matter for the defender. First, the attack leaves every validation number untouched: the test AUROC above was computed on genuine data and does not change, while the sensitivity for the attacked episodes falls to the share of attacks that failed. Second, a plausibility filter on the rate of change does not help much, because the manipulated traces stay physiologically smooth.

In [ ]:
def max_rate(W):
    return np.abs(np.diff(np.atleast_2d(W), axis=1)).max(axis=1) / 5.0


W_att = np.vstack(att.loc[ok, "x_adv"].to_numpy()) if ok.any() else np.empty((0, PAST))
print(f"sensitivity on the attacked episodes: before 100%, after {1 - ok.mean():.0%}")
print(f"share of traces with a rate of change above 4 mg/dL per minute: "
      f"genuine {np.mean(max_rate(np.vstack(att['x0'])) > 4):.0%}, manipulated {np.mean(max_rate(W_att) > 4):.0%}")

## 4. What explanations reveal

**SHAP anomaly score.** A manipulated window receives the "no alert" decision, but it reaches that decision in an unusual way: the attacker changes whatever the model is most sensitive to, and the explanation records that pattern. The detector below learns the typical SHAP pattern of genuine windows that were close to the threshold without triggering it, and scores new windows by their Mahalanobis distance from that pattern. Using near-threshold windows as the reference is essential; otherwise the detector would only learn that manipulated windows have a risk just below the threshold. The same distance computed on the raw features serves as a baseline.

In [ ]:
band = np.where((y_te == 0) & (p_te >= 0.5 * tau) & (p_te < tau))[0]
band = rng.permutation(band)[:3000]
half = len(band) // 2
F_ref_fit, F_ref_test = cgm_features(Xw_te[band[:half]]), cgm_features(Xw_te[band[half:]])
F_att = cgm_features(W_att)
S_ref_fit, S_ref_test, S_att = (tree_explainer.shap_values(F) for F in (F_ref_fit, F_ref_test, F_att))

shap_det = MahalanobisDetector().fit(S_ref_fit)
raw_det = MahalanobisDetector().fit(F_ref_fit)
s_clean, s_att = shap_det.score(S_ref_test), shap_det.score(S_att)
print(f"AUROC, SHAP anomaly score: {detection_auc(s_clean, s_att):.3f}")
print(f"AUROC, raw-feature distance: {detection_auc(raw_det.score(F_ref_test), raw_det.score(F_att)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
bins = np.linspace(0, np.quantile(np.concatenate([s_clean, s_att]), 0.99), 30)
axes[0].hist(s_clean, bins=bins, color=SAFE, alpha=0.7, density=True, label="genuine, near threshold")
axes[0].hist(s_att, bins=bins, color=ALERT, alpha=0.7, density=True, label="manipulated")
axes[0].set_xlabel("SHAP anomaly score")
axes[0].legend(frameon=False)
diff = pd.Series(S_att.mean(axis=0) - S_ref_test.mean(axis=0), index=FEATURE_NAMES).sort_values()
axes[1].barh(diff.index, diff.to_numpy(), color=[ALERT if v < 0 else SAFE for v in diff])
axes[1].set_xlabel("mean SHAP, manipulated minus genuine (log-odds)")
axes[1].set_title("Where the attack pushes the explanation")
plt.tight_layout()
plt.show()

**Counterfactual margin.** For every window with a "no alert" decision, the function below finds the smallest uniform decrease of all readings, up to 20 mg/dL, that would make the alert fire. In clinical words: how much lower would glucose have to be for the system to warn? A small margin means the silence hangs on a change smaller than sensor noise.

In [ ]:
def lower_all(W, s):
    return np.clip(W - 20.0 * np.asarray(s)[:, None], 40, 400)


m_clean = 20.0 * counterfactual_margin(decide, Xw_te[band[half:]], lower_all)
m_att = 20.0 * counterfactual_margin(decide, W_att, lower_all)
print(f"median counterfactual margin: genuine near-threshold {np.median(m_clean):.1f} mg/dL · "
      f"manipulated {np.median(m_att):.1f} mg/dL")
print(f"AUROC, counterfactual margin (smaller is more suspicious): {detection_auc(-m_clean, -m_att):.3f}")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
bins = np.linspace(0, 20, 41)
ax.hist(m_clean, bins=bins, color=SAFE, alpha=0.7, density=True, label="genuine, near threshold")
ax.hist(m_att, bins=bins, color=ALERT, alpha=0.7, density=True, label="manipulated")
ax.set_xlabel("decrease needed to trigger the alert (mg/dL)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**GEMEX geodesic view.** GEMEX [9] explains the model directly on the twelve raw readings. The geodesic length measures how far, in the Fisher information geometry of the model's output, the input lies from the reference population. Manipulated windows sit in a part of the input space that the attacker reached by following the decision boundary, and the geodesic length may reflect that. The comparison below uses the same near-threshold genuine windows as reference.

In [ ]:
READING_NAMES = [f"t-{5 * (PAST - 1 - k)}min" for k in range(PAST)]
gx = Explainer(alert_model, data_type="tabular", feature_names=READING_NAMES, class_names=["no alert", "alert"])
reference = Xw_tr[rng.choice(len(Xw_tr), size=300, replace=False)]
n_g = min(4 if QUICK else 15, len(W_att))
geo_clean = [gx.explain(Xw_te[i], X_reference=reference).geodesic_lengths[-1] for i in band[half:half + n_g]]
geo_att = [gx.explain(w, X_reference=reference).geodesic_lengths[-1] for w in W_att[:n_g]]
print(f"median geodesic length: genuine {np.median(geo_clean):.3f} · manipulated {np.median(geo_att):.3f} · "
      f"AUROC {detection_auc(geo_clean, geo_att):.3f} (n = {n_g} each)")
res_att = gx.explain(W_att[0], X_reference=reference)
res_att.plot("gsf_bar", theme="light")
plt.show()

**A campaign, not a single reading.** An attacker who silences one alert is dangerous; an attacker who silences alerts for days is a campaign. The Page-Hinkley test watches a stream of scores from the near-threshold windows, where manipulated windows would appear. Its threshold is calibrated on streams without any attack, and each stream below switches halfway to a state in which a share of the windows is manipulated. The same test runs on both signals.

In [ ]:
plot_campaigns([("SHAP anomaly score", s_clean, s_att),
                ("counterfactual margin (sign flipped)", -m_clean, -m_att)])

**What the evidence supports.** A decision-based attack makes the smallest change that works, so the attribution pattern of a manipulated window can look ordinary, and the SHAP anomaly score may separate it only weakly; compare the printed AUROC values. The same minimality is the attack's weakness: it ends on the decision boundary, and the counterfactual margin, an explanation of how close the decision is to flipping, exposes that. Explanations also support forensics at the population level, as the attribution shift plot shows where the attack pushed the model. None of this is a guarantee. An adaptive attacker can buy a larger margin with larger changes, which in turn enlarge the attribution anomaly, and every detector threshold also produces false alarms. The defensible claim is that explanations give the defender signals that accuracy metrics cannot give at all.

## 5. Try it

Move the slider from the genuine trace (0) to the manipulated trace (1) and watch the risk cross the threshold.

In [ ]:
def blend(example=0, alpha=1.0):
    r = att[ok].iloc[int(example)]
    w = r["x0"] + alpha * (r["x_adv"] - r["x0"])
    p = alert_model.predict_proba(w[None])[0, 1]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(t_past, r["x0"], "o-", color=MUTED, ms=3, label="true CGM")
    ax.plot(t_past, w, "o-", color=ALERT if p < tau else INK, ms=3, label="reported CGM")
    ax.plot(t_future, fut_te[r["index"]], "s--", color=AMBER, ms=3, label="what happened next")
    ax.axhline(70, color=ALERT, lw=0.8, ls=":")
    ax.set_title(f"risk {p:.3f} vs threshold {tau:.3f}: {'ALERT' if p >= tau else 'silent'}")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()


if ok.any():
    widgets.interact(blend, example=widgets.IntSlider(value=0, min=0, max=int(ok.sum()) - 1, continuous_update=False),
                     alpha=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, continuous_update=False));

## Discussion

1. The manipulation stays within the accuracy that CGM users already accept. What does that imply for the regulatory notion of robustness in Article 15 of the EU AI Act [12]?
2. Which party would operate the SHAP anomaly detector in practice: the device manufacturer, the phone application, or the clinic?
3. How could the detector itself be attacked, and what would that cost the attacker?

## Summary

On real CGM data from older adults with type 1 diabetes, a decision-based attacker who sees nothing but the alert state can silence correct hypoglycaemia alerts with changes of at most 15 mg/dL per reading, while the model's validation performance remains exactly as reported. Explanations make the manipulation observable, although not every explanation signal is equally informative: the attribution pattern of a minimal manipulation can look ordinary, while the counterfactual margin reveals that the silence hangs on a change smaller than sensor noise, and a sequential test turns such signals into a campaign alarm. NB5 moves to the intensive care unit, where the attacker also sees the risk score.

## References

[1] J. Chen, M. I. Jordan, and M. J. Wainwright, "HopSkipJumpAttack: A query-efficient decision-based attack," in *Proc. IEEE Symposium on Security and Privacy (SP)*, 2020, pp. 1277–1294. doi: 10.1109/SP40000.2020.00045

[2] E. S. Page, "Continuous inspection schemes," *Biometrika*, vol. 41, no. 1/2, pp. 100–115, 1954. doi: 10.1093/biomet/41.1-2.100

[3] W. Brendel, J. Rauber, and M. Bethge, "Decision-based adversarial attacks: Reliable attacks against black-box machine learning models," in *Proc. Int. Conf. Learning Representations (ICLR)*, 2018. arXiv:1712.04248

[4] R. S. Weinstock et al., "Risk factors associated with severe hypoglycemia in older adults with type 1 diabetes," *Diabetes Care*, vol. 39, no. 4, pp. 603–610, 2016. doi: 10.2337/dc15-1426

[5] R. Sergazinov, E. Chun, V. Rogovchenko, N. Fernandes, N. Kasman, and I. Gaynanova, "GlucoBench: Curated list of continuous glucose monitoring datasets with prediction benchmarks," in *Proc. Int. Conf. Learning Representations (ICLR)*, 2024. Data and code: https://github.com/IrinaStatsLab/GlucoBench

[6] T. Battelino et al., "Clinical targets for continuous glucose monitoring data interpretation: Recommendations from the international consensus on time in range," *Diabetes Care*, vol. 42, no. 8, pp. 1593–1603, 2019. doi: 10.2337/dci19-0028

[7] S. M. Lundberg and S.-I. Lee, "A unified approach to interpreting model predictions," in *Advances in Neural Information Processing Systems 30*, 2017, pp. 4765–4774. arXiv:1705.07874

[8] S. M. Lundberg et al., "From local explanations to global understanding with explainable AI for trees," *Nature Machine Intelligence*, vol. 2, pp. 56–67, 2020. doi: 10.1038/s42256-019-0138-9

[9] U. Kose, "GEMEX: Geodesic Entropic Manifold Explainability," Python package, version 1.2.2, 2026. https://pypi.org/project/gemex/

[10] S. G. Finlayson, J. D. Bowers, J. Ito, J. L. Zittrain, A. L. Beam, and I. S. Kohane, "Adversarial attacks on medical machine learning," *Science*, vol. 363, no. 6433, pp. 1287–1289, 2019. doi: 10.1126/science.aaw4399

[11] U.S. Food and Drug Administration, "FDA warns patients and health care providers about potential cybersecurity concerns with certain Medtronic insulin pumps," safety communication, 27 June 2019.

[12] Regulation (EU) 2024/1689 (Artificial Intelligence Act), Article 15, *Official Journal of the European Union*, L series, 12 July 2024.